# Test simulation du sign-unwrapping ODMR/NV

Ce notebook simule une carte de champ magnétique, calcule la projection NV signée ground truth, ne garde que $|B_\parallel|$, puis teste :

1. annotation de seeds,
2. graph cut,
3. raffinement continu,
4. reconstruction Fourier du champ vectoriel,
5. comparaison à la ground truth.

Fichiers requis dans le même dossier que ce notebook :

- `odmr_sign_unwrap.py`
- `sim_odmr_field_helpers.py`


In [ ]:
%matplotlib inline
from pathlib import Path
ROOT = Path.cwd()
assert (ROOT / "helpers.py").is_file(), "Launch from the repository root."
OUT = ROOT / "outputs" / "simulation"
OUT.mkdir(parents=True, exist_ok=True)


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Imports
# ─────────────────────────────────────────────────────────────────────────────

from pathlib import Path
import sys
import importlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.titlesize"] = 10

# Le notebook suppose que les deux helpers sont dans le même dossier.
NOTEBOOK_DIR = Path.cwd()
sys.path.insert(0, str(NOTEBOOK_DIR))

import odmr_sign_unwrap as osu
import sim_odmr_field_helpers as simh

osu = importlib.reload(osu)
simh = importlib.reload(simh)

print("Loaded:")
print("  odmr_sign_unwrap.py")
print("  sim_odmr_field_helpers.py")
print("Working directory:", NOTEBOOK_DIR)


if getattr(osu, "maxflow", None) is None:
    print("WARNING: PyMaxflow is not installed. Graph cut will fail until you run: pip install PyMaxflow")


## 1. Simulation d'une seule carte magnétique

Ici on garde uniquement le cas `generate_spin_lattice`, qui correspond à la simulation de lattice classique utilisée pour produire la dernière carte $|B_\parallel|$.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Configuration + simulation
# ─────────────────────────────────────────────────────────────────────────────

cfg = simh.SimulationConfig(
    NX=110,
    NY=110,
    Lx=3e-6,
    Ly=3e-6,
    z_meas=100e-9,
    Ms=8e5,
    t_film=10e-9,
    theta_nv_deg=125.25,
    phi_nv_deg=90.0,
)

sim = simh.simulate_classic_lattice(cfg, spin_seed=4)
simh.print_simulation_summary(cfg, sim)

spins = sim["spins"]
Bx_gt, By_gt, Bz_gt = sim["Bx"], sim["By"], sim["Bz"]
Bpar_gt_mT = sim["Bpar_gt_mT"]
A_sim_mT = sim["A_sim_mT"]
valid_mask = np.isfinite(A_sim_mT)

B_gt = (Bx_gt, By_gt, Bz_gt)


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Plots de la simulation
# ─────────────────────────────────────────────────────────────────────────────

simh.plot_spin_lattice(spins, cfg, title="Spin lattice simulation")
plt.show()

simh.plot_bpar_abs(Bpar_gt_mT, A_sim_mT, cfg)
plt.savefig(OUT / "simu_NV_projection_lattice_abs.pdf")
plt.show()


## 2. Seeds

In [ ]:
# Saved seeds provide the reproducible default; set True to annotate interactively.
USE_INTERACTIVE_SEEDS = False
if USE_INTERACTIVE_SEEDS:
    get_ipython().run_line_magic('matplotlib', 'widget')
    annot = osu.SeedAnnotator(A_sim_mT, radius=4, show_status=False)


In [ ]:
if USE_INTERACTIVE_SEEDS:
    seed_mask = annot.seed_mask.copy()
    osu.plot_seed_map(A_sim_mT, seed_mask, title="Manual seeds")
    plt.show()


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Sauvegarde du seed_mask
# ─────────────────────────────────────────────────────────────────────────────

# from pathlib import Path
# import numpy as np

# seed_dir = Path("saved_seed_masks")
# seed_dir.mkdir(exist_ok=True)

# seed_path = seed_dir / "seed_mask_simulation.npy"

# np.save(seed_path, seed_mask.astype(np.int8))

# print(f"seed_mask sauvegardé dans : {seed_path.resolve()}")
# print("shape :", seed_mask.shape)
# print("valeurs uniques :", np.unique(seed_mask))

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Chargement du seed_mask
# ─────────────────────────────────────────────────────────────────────────────
%matplotlib inline
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

USE_TRUTH_CHECKED_SEEDS = False  # Original manual annotations by default.
seed_filename = "seed_mask_simulation_checked.npy" if USE_TRUTH_CHECKED_SEEDS else "seed_mask_simulation.npy"
seed_path = ROOT / "saved_seed_masks" / seed_filename

if not USE_INTERACTIVE_SEEDS:
    seed_mask = np.load(seed_path).astype(np.int8)

print(f"seed_mask chargé depuis : {seed_path.resolve()}")
print("shape :", seed_mask.shape)
print("valeurs uniques :", np.unique(seed_mask))

# Vérification optionnelle
osu.plot_seed_map(A_sim_mT, seed_mask, title="Simulated field manual seeds")
plt.savefig(OUT / "seeds.pdf")
plt.show()

In [ ]:
#%matplotlib widget

#annot = osu.SeedAnnotator(
#    A_sim_mT,
#    seed_mask=seed_mask,
#    radius=4,
#    show_status=True
#)

In [ ]:
#%matplotlib widget

#annot = osu.SeedAnnotator(A_sim_mT, radius=4, show_status=False)
#annot.seed_mask[:] = seed_mask.copy()
#annot.redraw()

In [ ]:
## Récupérer le masque modifié depuis l'annotateur
#seed_mask = annot.seed_mask.copy()

## Sauvegarder dans le fichier initial
#np.save(seed_path, seed_mask.astype(np.int8))

#print(f"Seeds sauvegardées : {seed_path.resolve()}")
#print("Nombre de seeds restantes :", np.count_nonzero(seed_mask))

In [ ]:
%matplotlib inline

## 3. Graph cut + raffinement continu

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Sign-unwrapping par graph cut
# ─────────────────────────────────────────────────────────────────────────────

B_gc_mT, sign_gc, info_gc = osu.sign_unwrap_graphcut(
    A=A_sim_mT,
    seed_mask=seed_mask,
    valid_mask=valid_mask,
    lambda_pair=1.0,
    connectivity=4,
    normalize_percentile=95,
    seed_cost=1e6,
    verbose=True,
)

# En simulation seulement : corrige l'ambiguïté globale pour comparer à la GT.
B_gc_mT, global_sign_gc = simh.align_global_sign(B_gc_mT, Bpar_gt_mT)
sign_gc = np.sign(B_gc_mT)

print(f"Global sign graph cut = {global_sign_gc:+d}")

osu.plot_graphcut_result(
    A_sim_mT,
    B_gc_mT,
    sign_map=sign_gc,
    seed_mask=None, #seed_mask,
    title_prefix="Simulation (graph cut)",
)
plt.show()


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Raffinement continu initialisé par le graph cut
# ─────────────────────────────────────────────────────────────────────────────

B_ref_mT, info_ref = osu.refine_B_continuous(
    A=A_sim_mT,
    B_init=B_gc_mT,
    seed_mask=seed_mask,
    valid_mask=valid_mask,
    tau_mT=0.15,
    mu_data=1.0,
    lambda_lap=0.03,
    lambda_anchor=0.05,
    lambda_seed=10.0,
    data_weight_floor=0.05,
    maxiter=300,
    verbose=True,
)

# En simulation seulement : corrige l'ambiguïté globale pour comparer à la GT.
B_ref_mT, global_sign_ref = simh.align_global_sign(B_ref_mT, Bpar_gt_mT)

print(f"Global sign refined = {global_sign_ref:+d}")

osu.plot_refinement_comparison(
    A_sim_mT,
    B_gc_mT,
    B_ref_mT,
    seed_mask=None, #seed_mask,
    title="Simulation (graph cut + continuous refinement)",
)
plt.show()


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Comparaison projection NV : GT, |B|, graph cut, refined, erreur
# ─────────────────────────────────────────────────────────────────────────────

simh.plot_projection_comparison(
    Bpar_gt_mT=Bpar_gt_mT,
    #A_sim_mT=A_sim_mT,
    B_gc_mT=B_gc_mT,
    B_ref_mT=B_ref_mT,
    cfg=cfg,
    figsize=(9.5,8),
    title=None,
)
plt.savefig(OUT / "GT_graphcut.pdf")
plt.show()

rms_gc = np.sqrt(np.nanmean((B_gc_mT - Bpar_gt_mT) ** 2))
rms_ref = np.sqrt(np.nanmean((B_ref_mT - Bpar_gt_mT) ** 2))

print(f"RMS B_parallel graph cut : {rms_gc:.4g} mT")
print(f"RMS B_parallel refined   : {rms_ref:.4g} mT")

projection_error_metrics = pd.DataFrame([
    dict(method=method, RMS_mT=float(np.sqrt(np.mean((field-Bpar_gt_mT)**2))),
         max_error_mT=float(np.max(np.abs(field-Bpar_gt_mT))),
         max_GT_mT=float(np.max(np.abs(Bpar_gt_mT))))
    for method, field in [('graph cut', B_gc_mT), ('refined', B_ref_mT)]])
display(projection_error_metrics)


In [ ]:
import numpy as np

gt = np.asarray(Bpar_gt_mT if "Bpar_gt_mT" in globals()
                else sim["Bpar_gt_mT"])

gc = np.asarray(B_gc_mT)
ref = np.asarray(B_ref_mT)

assert gt.shape == gc.shape == ref.shape

for threshold in [0, 0.1, 0.5]:
    mask = (np.isfinite(gt) & np.isfinite(gc) & np.isfinite(ref)
            & (np.abs(gt) > threshold))

    if not mask.any():
        continue

    acc_gc = np.mean(np.sign(gc[mask]) == np.sign(gt[mask]))
    acc_ref = np.mean(np.sign(ref[mask]) == np.sign(gt[mask]))

    print(f"|B_gt| > {threshold:.1f} mT (N={mask.sum()}): "
          f"Graph-cut = {acc_gc:.3%}, Refined = {acc_ref:.3%}")

## 4. Reconstruction Fourier du champ vectoriel

Le mode DC n'est pas déterminable depuis une seule projection NV. Pour ce benchmark de simulation, on peut choisir d'ajouter la DC ground truth afin d'évaluer principalement la reconstruction spatiale. Pour un cas expérimental réel, laisser `USE_KNOWN_DC_FOR_SIMULATION = False`.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Reconstruction Fourier depuis le B_parallel reconstruit
# ─────────────────────────────────────────────────────────────────────────────

USE_KNOWN_DC_FOR_SIMULATION = False

known_dc_gt = (
    np.nanmean(Bx_gt),
    np.nanmean(By_gt),
    np.nanmean(Bz_gt),
)
known_dc = known_dc_gt if USE_KNOWN_DC_FOR_SIMULATION else None

B_rec = simh.reconstruct_vector_field_from_nv_projection(
    Bnv_T=B_ref_mT * 1e-3,
    dx=cfg.dx,
    nv_hat=cfg.nv_hat,
    d_meas=cfg.z_val, lambda0=1e-8, p=10, alpha=1.0,
    subtract_mean=True,
    known_dc=known_dc,
)

Bx_rec, By_rec, Bz_rec = B_rec


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Métriques RMS
# ─────────────────────────────────────────────────────────────────────────────

df_err = simh.make_error_dataframe(
    Bpar_rec_mT=B_ref_mT,
    Bpar_gt_mT=Bpar_gt_mT,
    B_rec=B_rec,
    B_gt=B_gt,
)

display(df_err)


In [ ]:
# Display AC fields consistently with the AC metrics.
B_gt_ac = tuple(b-np.nanmean(b) for b in B_gt)
B_rec_ac = tuple(b-np.nanmean(b) for b in B_rec)
# ─────────────────────────────────────────────────────────────────────────────
# Plots champ vectoriel : GT, reconstruction, erreur
# ─────────────────────────────────────────────────────────────────────────────

simh.plot_reconstruction_3D_comparison(
    B_gt=B_gt_ac,
    B_rec=B_rec_ac,
    cfg=cfg,
    error_arrow_gain=18.0,  # Relative to GT/reconstruction arrows.
    title_rec="Fourier reconstruction from refined $B_\parallel$",
)
plt.savefig(OUT / "reconstruction3D_GT_refined.pdf")
plt.show()

simh.plot_component_errors(
    B_rec=B_rec_ac,
    B_gt=B_gt_ac,
    cfg=cfg,
    title=None, #"Component errors after Fourier reconstruction",
)
plt.savefig(OUT / "error_BxByBz.pdf")
plt.show()


## 5. Paramètres à ajuster

Les paramètres les plus utiles à tester sont dans les cellules 5 et 6 :

- `lambda_pair` : régularisation du graph cut ;
- `tau_mT` : largeur de la zone autour de zéro pour le raffinement ;
- `lambda_lap` : lissage/courbure ;
- `lambda_anchor` : attachement au graph cut initial.


In [ ]:
# Report installed versions without modifying the project environment.
from importlib.metadata import version, PackageNotFoundError
for package in ('numpy', 'scipy', 'matplotlib', 'pandas', 'PyMaxflow'):
    try:
        print(f'{package}=={version(package)}')
    except PackageNotFoundError:
        print(f'{package}: unavailable')


The vector reconstruction uses the same filter as the experimental workflow:
lambda0=1e-8, p=10, alpha=1 and k0=alpha/z_val. Its three means are set to zero.
Full RMS therefore includes any true DC component; AC RMS removes the means of
both reference and reconstructed fields. Manual seeds and the optional global sign
alignment use simulation information and do not establish experimental absolute sign.
Maximum errors are computed from arrays, independently of plotting percentiles.
The original manual simulation mask is used by default. USE_TRUTH_CHECKED_SEEDS=True
selects a separate mask with 12 conflicting annotated pixels removed using truth;
that option is truth-assisted and cannot be described as blind manual annotation.
Error arrows are enlarged by a factor 18 relative to the first two panels; their
color scale and the numerical errors are unaffected. Experimental seeds are unchanged.


In [ ]:
# A cropped measurement window tests boundaries without retuning parameters.
fov_metrics = simh.finite_fov_diagnostic(cfg)
display(fov_metrics)


In [ ]:
# Export metrics and arrays independently of plotting limits.
projection_error_metrics.to_csv(OUT / 'projection_metrics.csv', index=False)
df_err.to_csv(OUT / 'vector_metrics.csv', index=False)
fov_metrics.to_csv(OUT / 'finite_fov_metrics.csv', index=False)
np.savez_compressed(OUT / 'simulation_results.npz',
    A_mT=A_sim_mT, Bpar_gt_mT=Bpar_gt_mT,
    B_gc_mT=B_gc_mT, B_ref_mT=B_ref_mT, seed_mask=seed_mask,
    Bx_gt=Bx_gt, By_gt=By_gt, Bz_gt=Bz_gt,
    Bx_rec=Bx_rec, By_rec=By_rec, Bz_rec=Bz_rec,
    global_sign_gc=global_sign_gc, global_sign_ref=global_sign_ref)
